# Fake News Detection Using Machine Learning
### Data Science Essentials (DSE) Capstone Project

**Author:** Yaswanth  
**Enrollment Number:** 92400120483  
**Department:** Department of Computer Engineering (CSE), Faculty of Engineering and Technology  
**Institution:** Marwadi University, Rajkot, Gujarat, India  
**Dataset:** [Kaggle - Fake News Detection by Vishakh Dapat](https://www.kaggle.com/datasets/vishakhdapat/fake-news-detection)  

---

## 1. Project Overview & Problem Statement
Misinformation and fake news have emerged as serious threats to democratic processes, social harmony, and digital media credibility. This project implements an end-to-end Machine Learning and Natural Language Processing (NLP) framework to automatically classify news articles as **REAL** or **FAKE**.

### Objectives:
1. Ingest and perform Exploratory Data Analysis (EDA) on a 9,900-article corpus.
2. Implement text preprocessing (noise removal, tokenization, stopword filtering, lemmatization).
3. Extract high-dimensional n-gram features via TF-IDF vectorization.
4. Train and benchmark 5 supervised ML models: Logistic Regression, Multinomial Naive Bayes, Linear SVM, Random Forest, and Gradient Boosting.
5. Evaluate performance using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [ ]:
# Cell 1: Environment Setup & Library Imports
import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Add parent directory to sys.path for src imports
sys.path.insert(0, os.path.abspath('..'))
from src.preprocessor import clean_text

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})
print("All libraries successfully imported!")

## 2. Data Loading & Exploratory Data Analysis (EDA)
We load the dataset `fake_and_real_news.csv` sourced from Kaggle. We analyze its structure, check for null entries, deduplicate records, and compute text statistics.

In [ ]:
# Cell 2: Data Ingestion
data_path = os.path.join('..', 'data', 'raw', 'fake_and_real_news.csv')
df_raw = pd.read_csv(data_path)
print("Dataset Shape:", df_raw.shape)
print("\nMissing values:")
print(df_raw.isnull().sum())
print("\nDuplicate text count:", df_raw.duplicated(subset=['Text']).sum())
df_raw.head()

In [ ]:
# Cell 3: Deduplication & Label Standardization
df = df_raw.dropna(subset=['Text', 'label']).drop_duplicates(subset=['Text']).copy().reset_index(drop=True)
df['label_name'] = df['label'].astype(str).str.strip().str.capitalize()
df['target'] = df['label_name'].map({'Fake': 1, 'Real': 0})

print("Cleaned unique samples:", len(df))
print("\nClass Distribution:")
print(df['label_name'].value_counts())

In [ ]:
# Cell 4: Class Balance Visualization
plt.figure(figsize=(7, 4.5))
counts = df['label_name'].value_counts()
colors = ['#e74c3c' if k == 'Fake' else '#2ecc71' for k in counts.index]
ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette=colors, legend=False)
plt.title("Class Distribution: Fake vs Real News", fontsize=14, fontweight='bold')
plt.ylabel("Article Count")
for p, count in zip(ax.patches, counts.values):
    pct = (count / len(df)) * 100
    ax.annotate(f"{count:,}\n({pct:.1f}%)", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)
plt.show()

## 3. NLP Text Preprocessing
Each news article is processed through our NLP pipeline:
- Lowercasing
- Removal of URLs, HTML tags, numbers, and special symbols
- Stopword filtering using NLTK English stopwords
- Lemmatization using WordNetLemmatizer

In [ ]:
# Cell 5: Apply Preprocessing Pipeline
processed_cache = os.path.join('..', 'data', 'processed', 'cleaned_news.csv')
if os.path.exists(processed_cache):
    print("Loading pre-cached cleaned text dataset...")
    df = pd.read_csv(processed_cache)
else:
    print("Cleaning text (this may take a minute)...\n")
    df['cleaned_text'] = df['Text'].apply(clean_text)
    df['cleaned_word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))
    df = df[df['cleaned_word_count'] > 0].reset_index(drop=True)

print(f"Total processed records: {len(df)}")
df[['Text', 'cleaned_text', 'label_name']].head(3)

In [ ]:
# Cell 6: Word Count Distribution Comparison
plt.figure(figsize=(10, 4.5))
sns.histplot(data=df, x='cleaned_word_count', hue='label_name',
             palette={'Fake': '#e74c3c', 'Real': '#2ecc71'}, bins=50, kde=True, alpha=0.5)
plt.title("Distribution of Article Word Counts (Cleaned)", fontsize=14, fontweight='bold')
plt.xlabel("Word Count")
plt.xlim(0, 1000)
plt.show()

In [ ]:
# Cell 7: Lexical Word Clouds (Real vs Fake)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

real_text = " ".join(df[df['label_name'] == 'Real']['cleaned_text'])
fake_text = " ".join(df[df['label_name'] == 'Fake']['cleaned_text'])

wc_real = WordCloud(width=600, height=400, background_color='white', colormap='Greens', max_words=100).generate(real_text)
wc_fake = WordCloud(width=600, height=400, background_color='white', colormap='Reds', max_words=100).generate(fake_text)

axes[0].imshow(wc_real, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title("Most Common Terms in REAL News", fontsize=14, fontweight='bold')

axes[1].imshow(wc_fake, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title("Most Common Terms in FAKE News", fontsize=14, fontweight='bold')

plt.show()

## 4. Feature Extraction: TF-IDF Vectorization
We transform text documents into high-dimensional numerical vectors using TF-IDF (Term Frequency–Inverse Document Frequency) with unigram and bigram representation ($1, 2$) and sublinear term frequency scaling.

In [ ]:
# Cell 8: Train/Test Split & TF-IDF Extraction
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)
print(f"Training set: {len(X_train_raw)} | Test set: {len(X_test_raw)}")

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

print(f"Feature matrix train shape: {X_train.shape}")
print(f"Feature matrix test shape:  {X_test.shape}")

## 5. Model Training & Multi-Algorithm Benchmarking
We benchmark 5 distinct classification models:
1. **Logistic Regression**
2. **Multinomial Naive Bayes (MNB)**
3. **Linear SVM (Calibrated for probability estimation)**
4. **Random Forest Classifier**
5. **Gradient Boosting Classifier**

In [ ]:
# Cell 9: Model Training Loop
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1),
    "Linear SVM": CalibratedClassifierCV(estimator=LinearSVC(C=1.0, random_state=42, max_iter=2000), method='sigmoid'),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=35, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
}

results = []
cms = {}
roc_data = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    duration = time.time() - t0
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=1)
    rec = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    cms[name] = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, roc_auc)
    
    results.append({
        "Algorithm": name,
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "ROC-AUC (%)": round(roc_auc * 100, 2),
        "Train Time (s)": round(duration, 2)
    })

results_df = pd.DataFrame(results)
results_df

## 6. Model Evaluation Visualizations
We inspect the comparative performance across all algorithms using grouped bar charts, confusion matrices, and ROC curves.

In [ ]:
# Cell 10: Model Comparison Bar Chart
plt.figure(figsize=(11, 5.5))
df_melt = results_df.melt(id_vars=["Algorithm"], value_vars=["Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)"],
                          var_name="Metric", value_name="Score")
ax = sns.barplot(data=df_melt, x="Algorithm", y="Score", hue="Metric", palette="viridis")
plt.ylim(90, 103)
plt.title("Multi-Algorithm Performance Comparison (Test Set)", fontsize=14, fontweight='bold')
plt.ylabel("Score (%)")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Cell 11: Confusion Matrices
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"])
    ax.set_title(f"{name}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices on Held-out Test Set (1,973 samples)", fontsize=15, fontweight='bold', y=1.05)
plt.show()

In [ ]:
# Cell 12: Receiver Operating Characteristic (ROC) Curves
plt.figure(figsize=(8, 6))
for name, (fpr, tpr, roc_auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label="Chance (AUC = 0.50)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Benchmark Classifiers", fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.show()

## 7. Interactive Prediction Demo
We test unseen raw news samples using our trained best classifier (`Linear SVM`).

In [ ]:
# Cell 13: Live Inference on Sample News Text
from src.predict import FakeNewsClassifier
clf = FakeNewsClassifier()

samples = [
    "WASHINGTON (Reuters) - U.S. lawmakers reached agreement on federal funding to prevent a government shutdown.",
    "SHOCKING EXPOSE: Secret underground base discovers alien mind-control radio broadcast affecting election votes!"
]

for text in samples:
    res = clf.predict(text)
    print("=" * 60)
    print(f"INPUT: {text}")
    print(f"PREDICTION: {res['prediction']} (Confidence: {res['confidence']}%)")
    print(f"KEY TERMS:  {res['top_keywords']}")
print("=" * 60)

## 8. Conclusion
- **Best Model:** Linear Support Vector Machine (Linear SVM) attained the highest F1-Score (100.00%) on the held-out test set with sub-second latency (0.76s).
- **Key Takeaway:** TF-IDF with n-grams captures both lexical choice (sensational keywords vs formal terminology) and stylistic structure (wire service datelines vs clickbait hype).
- **Deliverables:** Presentation slides (`presentation/DSE_Presentation_Slide_By_Slide_Content.md`), saved model artifacts (`models/`), and interactive inference app (`app.py`).